In [ ]:
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, roc_auc_score
from spafe.features.mfcc import mfcc, imfcc
from spafe.features.bfcc import bfcc
from spafe.features.lfcc import lfcc
from spafe.features.lpc import lpc, lpcc
from spafe.features.msrcc import msrcc
from spafe.features.ngcc import ngcc
from spafe.features.psrcc import psrcc
from spafe.features.rplp import plp, rplp
from spafe.features.gfcc import gfcc

In [ ]:
SP = ['spectral_centroid', 'spectral_skewness', 'spectral_kurtosis', 'spectral_entropy', 'spectral_spread', 'spectral_flatness', 'spectral_rolloff',
      'spectral_flux', 'spectral_mean', 'spectral_rms', 'spectral_std', 'spectral_variance']
SPECTRAL_COMPLEX_VALUES = ['spectral_flatness', 'spectral_centroid', 'spectral_spread']
SPECTRUM_FEATURES = ['mfcc', 'bfcc', 'lfcc', 'lpc', 'lpcc', 'msrcc', 'ngcc', 'psrcc', 'plp', 'rplp', 'gfcc']
SPECTRUM_FEATURES_FUNCTIONS = [mfcc, bfcc, lfcc, lpc, lpcc, msrcc, ngcc, psrcc, plp, rplp, gfcc]
DROP_FEATURES = ["label", "duration", "size", "spectral_bandwidth"]

In [ ]:
df = pd.read_csv('../data/train.csv')

In [ ]:
file_paths = df['path'].values
labels = df['label'].apply(lambda x: 1 if x == 'real' else 0).values

In [ ]:
real_file_path = df[df['label'] == 'real']['path'].values[0]
fake_file_path = df[df['label'] == 'fake']['path'].values[0]

In [ ]:
def load_and_match_length(file_path_1, file_path_2, sr=16000):
    y1, sr1 = librosa.load(file_path_1, sr=sr)
    y2, sr2 = librosa.load(file_path_2, sr=sr)
    min_len = min(len(y1), len(y2))
    y1 = y1[:min_len]
    y2 = y2[:min_len]
    return (y1, sr1), (y2, sr2)

In [ ]:
def extract_sp_feats(y, sr, dtype="float64"):
    sp_feats = librosa.feature.spectral_contrast(y=y, sr=sr)
    sp_feats_dict = {}
    for sp_feat_name, sp_feat in zip(SP, sp_feats):
        sp_feats_dict[sp_feat_name] = sp_feat.mean() if len(sp_feat) > 0 else 0
        if sp_feat_name in SPECTRAL_COMPLEX_VALUES:
            sp_feats_dict[sp_feat_name] = np.array(sp_feat).real.mean()
    return sp_feats_dict

In [ ]:
def extract_spectrum_data(y, sr):
    spectrum_dict = {}
    spectrum_dict["signal"] = y.mean()
    for idx, feature_name in enumerate(SPECTRUM_FEATURES):
        feature = SPECTRUM_FEATURES_FUNCTIONS[idx](sig=y, fs=sr)
        if isinstance(feature, (np.ndarray, list)):
            feature = np.array(feature)
            if feature.ndim == 2:
                spectrum_dict[feature_name] = feature.mean()
            else:
                spectrum_dict[feature_name] = feature.mean()
        else:
            spectrum_dict[feature_name] = feature
    return spectrum_dict


In [ ]:
def filter_features(features):
    filtered_features = {}
    for f in features:
        if f not in DROP_FEATURES:
            filtered_features[f] = features[f]
    return list(filtered_features.values())

In [ ]:
def get_all_features_from_sample(y, sr):
    sp_feats = extract_sp_feats(y, sr)
    spectrum_data = extract_spectrum_data(y, sr)
    features = {**sp_feats, **spectrum_data}
    return filter_features(features)

In [ ]:
# 모든 파일에 대해 특징 추출
from tqdm import tqdm
features_list = []
for path in tqdm(file_paths):
    y, sr = librosa.load(path, sr=16000)
    features = get_all_features_from_sample(y, sr)
    features_list.append(features)

In [ ]:
features_df = pd.DataFrame(features_list)
features_df['label'] = labels

In [ ]:
X = features_df.drop(columns=['label'])
y = features_df['label']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

In [ ]:
report = classification_report(y_test, y_pred)
report

In [ ]:
visualize_high_frequency_spectrogram(real_y, real_sr, 'Real Audio High Frequency Spectrogram')
visualize_high_frequency_spectrogram(fake_y, fake_sr, 'Fake Audio High Frequency Spectrogram')